##Youtube视频总结

In [ ]:
%pip install youtube_transcript_api==0.6.2
%pip install openai
%pip install unstructured==0.14.2
%pip install tiktoken==0.6.0
%pip install langchain==0.1.16
%pip install openai==1.19.0
%pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
base_url = os.getenv("OPENAI_BASE_URL")
openai_api_key = os.getenv("OPENAI_API_KEY")
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter
from langchain.document_loaders import TextLoader
from langchain.chains.summarize import load_summarize_chain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.prompts import ChatPromptTemplate,SystemMessagePromptTemplate,HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
if not openai_api_key:
    raise ValueError("OpenAI API key not found in environment variables")
print(openai_api_key)
print(base_url)

In [ ]:
transcript = YouTubeTranscriptApi.get_transcript('reUZRyXxUs4')
formatter = TextFormatter()
text_formatted = formatter.format_transcript(transcript)
with open('transcript.txt','w',encoding='utf-8') as text_file:
    text_file.write(text_formatted)

In [ ]:
loader = TextLoader("./transcript.txt")
docs = loader.load()
print(f"you have {len(docs)} document(s) in your data")
print("There are {} characters in your document".format(len(docs[0].page_content)))

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap=0)
split_text = text_splitter.split_documents(docs)
print(f"you have {len(split_text)} split document")

In [ ]:
youtube_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("According to user's requirements To summarize this context :{context} and then returns it to the user"),
    HumanMessagePromptTemplate.from_template("requirements")
])
llm = ChatOpenAI(temperature=0,verbose=True,openai_api_key=openai_api_key)

In [ ]:
youtube_chain = load_summarize_chain(llm,chain_type="map_reduce",verbose=True)
input_docs = split_text
youtube_chain.run(input_documents=input_docs)

In [ ]:
youtube_chain = youtube_prompt | llm

In [ ]:
response = youtube_chain.invoke(input={
    "context": docs,
    "requirements": "Answer me use Chinese and simplify it",
})
print(f"SUMMARIZE ：{response.content}")